# Do-as-I-Do · Reconstruction — Google Colab

End-to-end hand + object **reconstruction and 6-DoF pose tracking** from a single demo video,
running the [`reconstruction/`](https://github.com/malik-group/do-as-i-do) pipeline on Colab.

### Requirements before you run
1. **GPU runtime.** The pipeline needs **>= 32 GB VRAM**. Select **Runtime > Change runtime type > A100 40GB**
   (or H100 if available on your Colab plan). A **T4 / L4 (16-24 GB) will not work** and will OOM in Stage 2/3.
2. **HuggingFace access.** You need access to the gated repos `facebook/sam-3d-objects` and `facebook/sam3`.
   Store your token as a Colab Secret named **`HF_TOKEN`** (key icon in the left sidebar).
3. **MANO hand models** (license-gated, manual download from https://mano.is.tue.mpg.de).
   Put `MANO_RIGHT.pkl` and `MANO_LEFT.pkl` in a folder on your Google Drive and point `MANO_DRIVE_DIR` below at it.
4. **Demo video** somewhere on your Google Drive; set `VIDEO_PATH` below (Drive-relative).

### What this notebook does
- Installs Miniconda and builds the pipeline's **4 conda envs** (`sam3`, `sam3d`, `hawor`, `tapnet`).
- Clones the repo with submodules, fetches all model weights.
- Places MANO models from Drive.
- Lets you **click the object** on the reference frame (Colab-friendly JS widget, replacing the X-display click GUI).
- Runs `run_pipeline.sh` end-to-end with the click substituted by your points.

Run cells top-to-bottom. Cells tagged **[setup]** only need to run once per session; **[run]** cells are per-video.

## 0 · Configuration  [run]

Edit these for your video. Everything else is derived.

In [ ]:
# @title Inputs { run: "auto", display-mode: "form" }
from google.colab import drive  # noqa: E402  (mounted properly in the next cell)

# --- Your video (path is RELATIVE to your Google Drive root) ---
VIDEO_DRIVE_PATH = "do_as_i_do/whisking.mp4"  # @param {type:"string"}

# --- Reference frame + object + anchor hand (same args as run_pipeline.sh) ---
FRAME_N = 125        # @param {type:"integer"}
OBJECT  = "whisk"    # @param {type:"string"}
ANCHOR_HAND = "right"  # @param ["right", "left"]

# --- MANO models on Drive (folder containing MANO_RIGHT.pkl + MANO_LEFT.pkl) ---
MANO_DRIVE_DIR = "do_as_i_do/mano"  # @param {type:"string"}

# --- Where to clone the repo (local VM disk, fast IO) ---
REPO_DIR = "/content/do-as-i-do"

# --- Where the video will be copied to (local VM disk; pipeline writes outputs next to it) ---
WORK_DIR = "/content/videos"

DRIVE_ROOT = "/content/drive/MyDrive"
print("Config loaded. Edit the values above, then run the next cells.")

## 1 · GPU & disk sanity check  [setup]

Asserts that the runtime actually has a >= 32 GB GPU and enough local disk for envs + weights.

In [ ]:
%%bash
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
VRAM_MB=$(nvidia-smi --query-gpu=memory.total --format=csv,noheader,nounits | head -n1 | tr -d ' ')
if [ "$VRAM_MB" -lt 30000 ]; then
  echo "!! This GPU has only ${VRAM_MB} MB VRAM. The pipeline needs >= 32 GB."
  echo "   Switch runtime to A100 / H100 and re-run."
fi
echo "--- local disk ---"
df -h /content | tail -n1

## 2 · Mount Google Drive + HuggingFace login  [setup]

In [ ]:
from google.colab import drive, userdata
drive.mount("/content/drive")

try:
    HF_TOKEN = userdata.get("HF_TOKEN")
    print("HF_TOKEN loaded from Colab Secrets.")
except Exception:
    HF_TOKEN = None
    print("!! No HF_TOKEN in Colab Secrets. Stage 1 (sam3) and Stage 2 (sam-3d-objects) will fail without it.")

# Expose to all subsequent bash cells.
import os
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN

In [ ]:
%%bash
pip install -q 'huggingface-hub[cli]<1.0'
# Use a persistent credential helper so we don't have to re-auth within this session.
git config --global credential.helper store
if [ -n "$HF_TOKEN" ]; then
  hf auth login --token "$HF_TOKEN" --add-to-git-credential
else
  echo "No HF_TOKEN set; skipping login (see cell above)."
fi

## 3 · Install Miniconda  [setup]

Colab has no `conda`. We install Miniconda to `/opt/conda`; every later `%%bash` cell re-sources it
(shell state does not persist between cells).

In [ ]:
%%bash
set -e
if [ -x /opt/conda/bin/conda ]; then
  echo "Miniconda already installed at /opt/conda"
else
  echo "Installing Miniconda..."
  cd /content
  wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O miniconda.sh
  bash miniconda.sh -bfp /opt/conda
  rm miniconda.sh
fi
source /opt/conda/etc/profile.d/conda.sh
conda --version
conda init bash >/dev/null 2>&1 || true
# Recent conda builds refuse to create envs until the defaults channels' ToS are accepted.
conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main || true
conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r || true

## 4 · Clone the repo + submodules  [setup]

`GIT_LFS_SKIP_SMUDGE=1` so the heavy weight blobs are **not** pulled by Git LFS — they come from
`setup/02_fetch_weights.sh` later.

In [ ]:
%%bash
set -e
if [ -d /content/do-as-i-do/.git ]; then
  echo "Repo already cloned at /content/do-as-i-do"
else
  cd /content
  GIT_LFS_SKIP_SMUDGE=1 git clone --recurse-submodules https://github.com/malik-group/do-as-i-do.git
fi
# Make sure submodules are initialised even if the clone was non-recursive.
cd /content/do-as-i-do
GIT_LFS_SKIP_SMUDGE=1 git submodule update --init --recursive
echo "--- submodule pins ---"
git submodule status

## 5 · Build the 4 conda envs  [setup]

**This notebook targets a Blackwell GPU (sm_120).** cu121 / cu117 builds will **not run** on it —
Blackwell needs CUDA ≥ 12.8. So:

- `sam3` and `tapnet` use `setup/01_create_envs.sh` as-is (those recipes already install cu128 torch).
- `sam3d` and `hawor` are built differently: their `env/*.yml` files *are* the known-good cu128
  sets (torch 2.8 / 2.9 +cu128), but the **conda** part of those full exports can't solve on Colab
  (glibc 2.39 / exact build-string pins). So we create a minimal conda env (python + ffmpeg + a
  CUDA 12.8 toolkit) and pip-install the **pip section** of the yml verbatim. The CUDA 12.8 toolkit
  is required so compiled extensions (pytorch3d, nvdiffrast, gsplat, lietorch, ...) target sm_120.

These two cells are slow (~30-45 min each) because of the source compiles. If one aborts, just
re-run **that** cell — pip skips already-satisfied packages and resumes.

In [ ]:
%%bash
set -e
source /opt/conda/etc/profile.d/conda.sh
cd /content/do-as-i-do/reconstruction
bash setup/01_create_envs.sh sam3

In [ ]:
%%bash
set -eo pipefail
source /opt/conda/etc/profile.d/conda.sh
cd /content/do-as-i-do/reconstruction

# === Blackwell (sm_120): the cu121 path (01_create_envs.sh sam3d) will NOT run here.
# env/sam3d.yml is the known-good cu128 set (torch 2.8+cu128) but its conda part can't solve
# on Colab, so: minimal conda env + pip-install the yml's pip section, with a CUDA 12.8
# toolkit present so compiled extensions (pytorch3d, nvdiffrast, gsplat, ...) build for sm_120. ===
if ! conda env list | grep -q '^sam3d '; then
  conda create -y -n sam3d python=3.11 -c conda-forge
fi
conda activate sam3d
conda install -y -c conda-forge ffmpeg
conda install -y -c "nvidia/label/cuda-12.8.0" cuda-toolkit   # nvcc 12.8 for sm_120 compiles
export CUDA_HOME="$CONDA_PREFIX"
export PATH="$CUDA_HOME/bin:$PATH"
export TORCH_CUDA_ARCH_LIST="12.0"    # Blackwell
export FORCE_CUDA=1
export MAX_JOBS=4                      # cap RAM during pytorch3d compile

# Pull the pip: section out of env/sam3d.yml; drop the editable/local pkg + dev noise
# (those either aren't on PyPI or aren't needed for inference).
awk '/^  - pip:/{f=1} f && /^      - /{sub(/^      - /,"");print}' env/sam3d.yml \
  | grep -vE '^(sam3d-objects==|conda-env-export==|findpydeps==|pipdeptree==|ruff==|yt-dlp==)' \
  > /tmp/sam3d_reqs.txt
echo "=== installing $(wc -l < /tmp/sam3d_reqs.txt) pip packages (compiled exts incl.; ~30-45 min) ==="
# Kaolin ships a prebuilt cp311/cu128/torch2.8 wheel on NVIDIA's bucket:
export PIP_FIND_LINKS="https://nvidia-kaolin.s3.us-east-2.amazonaws.com/torch-2.8.0_cu128.html"
pip install -r /tmp/sam3d_reqs.txt

# geocalib (yml predates it) + editable sam-3d-objects (filtered above) --no-deps to avoid
# re-resolving its cu121 metadata now that the env is fully populated.
pip install "geocalib @ git+https://github.com/cvg/GeoCalib.git"
pip install -e --no-deps modules/sam-3d-objects

echo "=== sam3d env ready ==="
python -c "import torch,pytorch3d,kaolin; print('torch',torch.__version__,'cuda',torch.version.cuda)"

In [ ]:
%%bash
set -eo pipefail
source /opt/conda/etc/profile.d/conda.sh
cd /content/do-as-i-do/reconstruction

# === Blackwell: 01_create_envs.sh hawor installs torch 1.13+cu117, which won't run on sm_120.
# Build from env/hawor.yml (torch 2.9+cu128) via minimal conda env + its pip section. ===
if ! conda env list | grep -q '^hawor '; then
  conda create -y -n hawor python=3.10 -c conda-forge
fi
conda activate hawor
conda install -y -c conda-forge ffmpeg
conda install -y -c "nvidia/label/cuda-12.8.0" cuda-toolkit   # nvcc 12.8 for sm_120 compiles
export CUDA_HOME="$CONDA_PREFIX"
export PATH="$CUDA_HOME/bin:$PATH"
export TORCH_CUDA_ARCH_LIST="12.0"    # Blackwell
export FORCE_CUDA=1
export MAX_JOBS=4

# Pull the pip: section out of env/hawor.yml; drop chumpy (0.71 not on PyPI, needs git
# + --no-build-isolation) and droid-backends (local, built from HaWoR thirdparty below).
awk '/^  - pip:/{f=1} f && /^      - /{sub(/^      - /,"");print}' env/hawor.yml \
  | grep -vE '^(chumpy==|droid-backends==)' \
  > /tmp/hawor_reqs.txt
echo "=== installing $(wc -l < /tmp/hawor_reqs.txt) pip packages (~20-35 min) ==="
pip install -r /tmp/hawor_reqs.txt

# chumpy 0.71 lives only on git (PyPI caps at 0.70) and needs numpy at build time.
pip install "chumpy@git+https://github.com/mattloper/chumpy" --no-build-isolation

# DROID-SLAM's bundled lietorch needs a 1-line patch to compile against torch>=2
# (::detail::scalar_type(the_type) -> the_type.scalarType()). Patch every copy we can find.
python - <<'PY'
import glob, os
roots = [
    os.path.join(os.environ["CONDA_PREFIX"], "lib"),
    "modules/HaWoR/thirdparty",
]
changed = []
for root in roots:
    for p in glob.glob(os.path.join(root, "**", "dispatch.h"), recursive=True):
        s = open(p).read()
        if "::detail::scalar_type(the_type)" in s:
            open(p, "w").write(s.replace("::detail::scalar_type(the_type)", "the_type.scalarType()"))
            changed.append(p)
print("patched dispatch.h:", changed or "none needed")
PY
( cd modules/HaWoR/thirdparty/DROID-SLAM && python setup.py install )

# torch>=2.6 loads checkpoints weights_only=True, which rejects HaWoR's omegaconf-bearing
# ckpts. Restore the pre-2.6 default for this env (set as a persistent env var).
conda env config vars set TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD=1 -n hawor

echo "=== hawor env ready ==="
python -c "import torch,lietorch,pytorch3d; print('torch',torch.__version__,'cuda',torch.version.cuda)"

In [ ]:
%%bash
set -e
source /opt/conda/etc/profile.d/conda.sh
cd /content/do-as-i-do/reconstruction
bash setup/01_create_envs.sh tapnet

## 6 · `sam3d` post-install fixes  [setup]

Two manual steps the repo's README calls out after building the `sam3d` env
([`env/README.md`](https://github.com/malik-group/do-as-i-do/blob/main/reconstruction/env/README.md)):

1. `pip uninstall -y notebook` so `notebook.inference` (vendored inside sam-3d-objects) imports cleanly.
2. Build the Mip-Splatting `diff_gaussian_rasterization` (needed by the `inria` GLB/texture baking backend).

Both are skipped silently if they have already been applied.

In [ ]:
%%bash
set -e
source /opt/conda/etc/profile.d/conda.sh
conda activate sam3d

# 1. un-shadow the repo's notebook/ package
pip uninstall -y notebook 2>/dev/null || true

# 2. Mip-Splatting gaussian rasterizer (only if not already importable)
python - <<'PY'
import importlib.util as u
if u.find_spec("diff_gaussian_rasterization") is None:
    import subprocess, os, shutil
    work = "/content/mip-splatting-build"
    if not os.path.isdir(work):
        subprocess.run(["git", "clone", "--recursive",
                        "https://github.com/autonomousvision/mip-splatting.git", work], check=True)
    sub = os.path.join(work, "submodules", "diff-gaussian-rasterization")
    env = os.environ.copy()
    env["CUDA_HOME"] = os.environ.get("CONDA_PREFIX")
    env["TORCH_CUDA_ARCH_LIST"] = "12.0"    # Blackwell (sm_120)
    env["FORCE_CUDA"] = "1"
    subprocess.run(["python", "setup.py", "install"], cwd=sub, env=env, check=True)
    print("diff_gaussian_rasterization built.")
else:
    print("diff_gaussian_rasterization already installed; skipping.")
PY

## 7 · Fetch model weights  [setup]

Runs the repo's `setup/02_fetch_weights.sh --download`. Pulls the **SAM3D** checkpoint set from
HuggingFace (gated), the **HaWoR** / **Metric3D** / **DROID-SLAM** checkpoints, and the
**BootsTAPIR** checkpoint, then symlinks the shared heavy SAM3D files into both module dirs.

The Stage-1 **SAM3** model is auto-downloaded by `run_sam3_video.py` at runtime (also gated).

In [ ]:
%%bash
set -eo pipefail
source /opt/conda/etc/profile.d/conda.sh
cd /content/do-as-i-do/reconstruction
bash setup/02_fetch_weights.sh --download

## 8 · Place MANO hand models from Drive  [setup]

MANO is license-gated and cannot be auto-downloaded. Copy the two `.pkl` files you downloaded from
https://mano.is.tue.mpg.de out of your Google Drive into the paths HaWoR expects.

In [ ]:
import os, shutil, sys

HAWOR = "/content/do-as-i-do/reconstruction/modules/HaWoR"
targets = {
    "MANO_RIGHT.pkl": f"{HAWOR}/_DATA/data/mano/MANO_RIGHT.pkl",
    "MANO_LEFT.pkl":  f"{HAWOR}/_DATA/data_left/mano_left/MANO_LEFT.pkl",
}
src_dir = os.path.join("/content/drive/MyDrive", MANO_DRIVE_DIR.strip("/"))

missing = []
for name, dst in targets.items():
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    src = os.path.join(src_dir, name)
    if os.path.isfile(src):
        shutil.copy2(src, dst)
        print(f"placed {name} -> {dst}")
    elif os.path.isfile(dst):
        print(f"{name} already present at {dst}")
    else:
        missing.append(src)

if missing:
    print("!! Could not find these MANO files (expected in Drive folder):")
    for m in missing:
        print("   ", m)
    sys.exit(1)
print("MANO models in place.")

## 9 · Setup sanity check  [setup]

Confirms all 4 envs exist and the key weights are where the pipeline expects them.

In [ ]:
%%bash
source /opt/conda/etc/profile.d/conda.sh
echo "=== conda envs ==="
conda env list
echo
echo "=== torch in each env ==="
for e in sam3 sam3d hawor tapnet; do
  echo -n "$e: "
  conda run -n "$e" python -c "import torch; print(torch.__version__, 'cuda=' + str(torch.version.cuda), 'avail=' + str(torch.cuda.is_available()))" 2>/dev/null || echo "(torch import failed)"
done
echo
echo "=== weight presence ==="
R=/content/do-as-i-do/reconstruction
ls -lh "$R/weights/tapnet/bootstapir_checkpoint_v2.pt" 2>/dev/null || echo "MISSING: tapnet ckpt"
ls -lh "$R/weights/sam3d_shared/hf/" 2>/dev/null | head
ls -lh "$R/modules/HaWoR/weights/hawor/checkpoints/hawor.ckpt" 2>/dev/null || echo "MISSING: hawor.ckpt"
ls -lh "$R/modules/HaWoR/_DATA/data/mano/MANO_RIGHT.pkl" 2>/dev/null || echo "MISSING: MANO_RIGHT.pkl"
echo
echo "Setup complete if nothing above says MISSING / failed."
echo "Then continue to the Run section."
R=/content/do-as-i-do/reconstruction

---
# Run phase (per-video)

Cells below are **[run]** — re-run them whenever you change `VIDEO_DRIVE_PATH` / `FRAME_N` / etc.

## 10 · Copy the video onto local disk  [run]

Reading the mp4 straight from Drive is slow (the pipeline reads it many times) and the pipeline
writes its outputs *next to* the video, so we copy it to `/content/videos/` first.

In [ ]:
import os, shutil, subprocess, json

src = os.path.join("/content/drive/MyDrive", VIDEO_DRIVE_PATH.strip("/"))
os.makedirs(WORK_DIR, exist_ok=True)
dst = os.path.join(WORK_DIR, os.path.basename(src))
if not os.path.isfile(src):
    raise FileNotFoundError(f"Video not found in Drive: {src}")
if not (os.path.isfile(dst) and os.path.getsize(dst) == os.path.getsize(src)):
    shutil.copy2(src, dst)
print("video at:", dst)

# Write these into the environment so later %%bash cells see them.
os.environ["VIDEO_LOCAL"] = dst
os.environ["VIDEO_DIR"]   = os.path.dirname(dst)
os.environ["FRAME_N"]     = str(FRAME_N)
os.environ["OBJECT"]      = OBJECT
os.environ["ANCHOR_HAND"] = ANCHOR_HAND
os.environ["REPO_DIR"]    = REPO_DIR

## 11 · Extract the reference frame for clicking  [run]

Uses the exact same `ffmpeg` invocation `run_pipeline.sh` will use in Step 0, so the frame
numbering matches the tracking output.

In [ ]:
%%bash
set -e
VIDEO_DIR="$(dirname "$VIDEO_LOCAL")"
mkdir -p "$VIDEO_DIR/all_frames"
# Same flags as run_pipeline.sh Step 0 (numbering starts at 0)
ffmpeg -y -i "$VIDEO_LOCAL" -vsync 0 -start_number 0 "$VIDEO_DIR/all_frames/%06d.png"
# Pull out the single reference frame for the click UI.
REF="$VIDEO_DIR/$(printf '%04d.png' "$FRAME_N")"
ffmpeg -y -i "$VIDEO_LOCAL" -vf "select=eq(n\,$FRAME_N)" -vsync 0 -vframes 1 "$REF"
echo "reference frame: $REF"
ls -la "$REF"

## 12 · Click the object on the reference frame  [run]

Colab has no X display, so the pipeline's OpenCV click GUI can't run. Instead, click directly on
the rendered frame below — each click adds a positive point (green dot). The pixel coordinates
are passed to `run_sam3_video.py` via `--points`, reproducing the `--click` flow headlessly.

**Click 1-3 points on the object you want to track** (e.g. the whisk head). Then run the next cell.

In [ ]:
import os, base64
from IPython.display import HTML, display
import google.colab.output as colab_output

REF_PNG = os.path.join(os.environ["VIDEO_DIR"], f"{int(os.environ['FRAME_N']):04d}.png")
with open(REF_PNG, "rb") as f:
    b64 = base64.b64encode(f.read()).decode()

clicks = []
def _on_click(coords):
    clicks.append((int(coords["x"]), int(coords["y"])))
    return f"point {len(clicks)}: {clicks[-1]}  (total {len(clicks)})"
colab_output.register_callback("_obj_click", _on_click)

display(HTML(f"""
<div>
  <img id="refimg" src="data:image/png;base64,{b64}"
       style="max-width:100%; cursor:crosshair; border:1px solid #444"/>
  <p id="info">Click on the <b>{os.environ['OBJECT']}</b>. Each click adds a point.</p>
</div>
<script>
(function(){{
  const img = document.getElementById('refimg');
  const info = document.getElementById('info');
  img.addEventListener('click', function(ev){{
    const r = img.getBoundingClientRect();
    const x = Math.round((ev.clientX - r.left) * (img.naturalWidth / r.width));
    const y = Math.round((ev.clientY - r.top)  * (img.naturalHeight / r.height));
    google.colab.kernel.invokeFunction('_obj_click', [{{x: x, y: y}}], {{}})
      .then(function(res){{ info.innerText = res.data['text/plain']; }});
  }});
}})();
</script>
"""))
print("Click the object on the image above, then advance to the next cell.")

In [ ]:
# Format the collected clicks into the --points / --point_labels args run_sam3_video.py expects.
assert len(clicks) >= 1, "Click at least one point on the object in the cell above first."
OBJ_POINTS = ";".join(f"{x},{y}" for x, y in clicks)
OBJ_POINT_LABELS = ";".join("1" for _ in clicks)   # all positive
print("Object points :", OBJ_POINTS)
print("Labels        :", OBJ_POINT_LABELS)

import os
os.environ["OBJ_POINTS"] = OBJ_POINTS
os.environ["OBJ_POINT_LABELS"] = OBJ_POINT_LABELS

## 13 · Run the full pipeline  [run]

We make a one-line patch to `run_pipeline.sh`: swap `--click` for a `--points`
/ `--point_labels` pair fed by the clicks you collected above. Everything else (env switching,
stages 0-4) is the repo's own driver, untouched.

In [ ]:
# Patch a copy of run_pipeline.sh: swap the single `--click` token for a --points/--point_labels
# pair, so Stage 1 runs headlessly using the clicks you just collected. The trailing backslash
# that followed `--click` in the original is left in place, so line continuation still works.
# (Python string replace avoids the shell-escaping nightmare of doing this in sed.)
import os
recon = os.path.join(os.environ["REPO_DIR"], "reconstruction")
src = open(f"{recon}/run_pipeline.sh").read()
assert src.count("--click") == 1, f"expected exactly one --click in run_pipeline.sh, found {src.count('--click')}"
repl = '--points "$OBJ_POINTS" --point_labels "$OBJ_POINT_LABELS"'
open(f"{recon}/run_pipeline_colab.sh", "w").write(src.replace("--click", repl))
print("wrote run_pipeline_colab.sh")

In [ ]:
%%bash
set -e
source /opt/conda/etc/profile.d/conda.sh
cd "$REPO_DIR/reconstruction"
chmod +x run_pipeline_colab.sh

# The patched object-segmentation line expands these at pipeline-run time.
export OBJ_POINTS="$OBJ_POINTS"
export OBJ_POINT_LABELS="$OBJ_POINT_LABELS"

echo "=== launching patched pipeline ==="
./run_pipeline_colab.sh "$VIDEO_LOCAL" "$FRAME_N" "$OBJECT" "$ANCHOR_HAND"

## 14 · Done — where the outputs live  [run]

All outputs are written next to the video under `$VIDEO_DIR/`. The directory consumed by the
[`retargeting/`](https://github.com/malik-group/do-as-i-do/tree/main/retargeting) pipeline is:

```
<VIDEO_DIR>/obj_tracking_out/<OBJECT>/combined_visualization/
    layout_camera_frame_optimized.json      <- final 6-DoF object pose track
    projected_*.png                          <- projected-mesh overlays
```

Also produced: per-object `.obj` meshes in `video_segmentation/masks/frame_NNNNNN_masks/<OBJECT>/`,
HaWoR `all_hand_meshes.npz`, per-frame `*_pointmap.npy` / `*_intrinsics.*`, and `gravity.json`.

In [ ]:
%%bash
VIDEO_DIR="$(dirname "$VIDEO_LOCAL")"
echo "=== tree of outputs ==="
find "$VIDEO_DIR" -maxdepth 3 -type d | sort
echo
echo "=== final layout json ==="
ls -la "$VIDEO_DIR/obj_tracking_out/$OBJECT/combined_visualization/layout_camera_frame_optimized.json" 2>/dev/null \
  && echo "OK: reconstruction finished successfully" \
  || echo "!! optimized layout json missing — check the pipeline log above."
echo
echo "To copy results back to Drive:"
echo "  cp -r \"$VIDEO_DIR\" \"/content/drive/MyDrive/<dest>\""